# Somo la 11 - Itifaki ya Muktadha ya Modeli (MCP)

**Itifaki ya Muktadha ya Modeli (MCP)** ni kiwango cha wazi kinachowezesha mawakala kugundua kwa wakati halisi na kutumia zana, rasilimali, na vyanzo vya data wakati wa utekelezaji. Badala ya kuingiza zana moja kwa moja katika wakala kwa msimbo thabiti, MCP inawawezesha mawakala kuunganishwa na seva za nje ambazo zinafunua uwezo wanapohitajika.

Katika somo hili, utajifunza:
- Nini MCP ni na kwa nini ni muhimu kwa mifumo ya mawakala
- Jinsi usanifu wa mteja-seva wa MCP unavyofanya kazi
- Jinsi ya kujenga mawakala wanaotumia ugunduzi wa zana kwa mtindo wa MCP


## Usanidi

**Mahitaji:**
- Mradi wa Azure AI Foundry wenye modeli iliyowekwa
- Endesha `az login` kwa ajili ya uthibitishaji


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Itifaki ya Muktadha wa Mfano (MCP) ni nini?

MCP inaeleza njia sanifu kwa mawakala wa AI kugundua na kuingiliana na zana za nje na vyanzo vya data:

- **MCP Server**: Inaonyesha zana, rasilimali, na maelekezo kupitia itifaki sanifu
- **MCP Client**: Mtekelezaji wa wakala unaounganisha na seva na kugundua uwezo unaopatikana
- **Ugunduzi Dinamiki**: Mawakala hawahitaji zana zilizowekwa kwenye msimbo — wanagundua vinavyopatikana wakati wa utekelezaji

Hii ni yenye nguvu kwa kujenga mifumo ya mawakala inayoweza kupanuliwa ambapo uwezo mpya unaweza kuongezwa bila kubadilisha msimbo wa wakala.


## Jinsi MCP Inavyofanya Kazi

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Wakala (MCP client) anajiunga na seva ya MCP
2. Seva inajibu kwa orodha ya zana zinazopatikana na miundo yao
3. Wakala anaweza kisha kuita chombo chochote alichokigundua wakati wa mchakato wake wa kufikiri
4. Matokeo yanarudi kupitia itifaki ile ile


## Kuiga Ugunduzi wa Zana za MCP

Kwa kuwa seva halisi ya MCP inahitaji mchakato wa seva unaoendelea, tutatoa mfano wa mtindo kwa kutumia kazi za `@tool` zinazofanya kuiga kile huduma ya malazi iliyounganishwa na MCP ingetoa.

Katika uzalishaji, zana hizi zingegunduliwa kiotomatiki kutoka kwa seva ya MCP badala ya kufafanuliwa kwa ndani.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Kujenga Wakala kwa Zana za Mtindo wa MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP katika Uzalishaji

Katika mazingira ya uzalishaji, MCP inaruhusu mifumo yenye nguvu:

- **Ugundaji wa zana kwa wakati wa utekelezaji**: Maajenti huunganishwa na seva za MCP na kugundua zana wakati wa utekelezaji
- **Usanifu uliotengwa**: Watoa zana wanaweza kusasisha kwa kujitegemea bila kutegemea maajenti
- **Kushirikisha taasisi mbalimbali**: Timu zinaweza kufichua uwezo kupitia seva za MCP ambazo maajenti yoyote yanaweza kutumia
- **Msaada wa Microsoft Agent Framework**: MAF ina msaada wa mteja wa MCP uliojengwa ndani kupitia ujumuishaji wa `mcp`

Ili kutumia seva halisi ya MCP na MAF, ungeunganishwa kupitia `hosted_mcp_tool()` au ujumuishaji wa mteja wa MCP.

**Jifunze zaidi:**
- [Ufafanuzi wa MCP](https://modelcontextprotocol.io/)
- [Msaada wa MCP wa Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Muhtasari

Katika somo hili, ulijifunza:
- **MCP** ni kiwango wazi cha kugundua zana kwa wakati wa utekelezaji kati ya maajenti na watoa zana
- Usanifu wa **mteja-seva** unawawezesha maajenti kugundua uwezo wakati wa utekelezaji
- MCP inawezesha **mifumo ya maajenti inayoweza kupanuliwa na isiyotegemeana** ambapo zana zinaweza kuongezwa bila mabadiliko ya msimbo
- Microsoft Agent Framework hutoa **msaada wa MCP uliojengwa ndani** kwa matumizi ya uzalishaji


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Tamko la kutokuwa na dhamana:
Nyaraka hii imefasiriwa kwa kutumia huduma ya utafsiri wa AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ijapo tunajitahidi kuwa sahihi, tafadhali fahamu kwamba utafsiri wa kiotomatiki unaweza kuwa na makosa au ukosefu wa usahihi. Nyaraka ya asili katika lugha yake ya asili inapaswa kuchukuliwa kama chanzo chenye mamlaka. Kwa taarifa muhimu, tunapendekeza tafsiri ya kitaalamu ya binadamu. Hatuwajibiki kwa kutoelewana au tafsiri zisizo sahihi zinazotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
